# Omega Profile vs IMERG Precipitation — Per-Circle Comparison

Calls `scripts/imerg_only_comparison.py` directly (no SSH).

**Figure layout per circle:**
- **Left** — Raw BEACH ω (gray dashed) + M4+ERA5 ω (coloured solid) vs pressure (hPa)
- **Right** — 2×2 grid of IMERG 30-min snapshots covering the actual circle flight window

Use the dropdown or set `CIRCLE` manually.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image

from scripts.imerg_only_comparison import (
    plot_one_circle,
    build_circle_sonde_index,
    imerg_times_as_np64,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
ZARR_PATH   = Path('/g/data/k10/zr7147/ORCESTRA_dropsondes_categorized.zarr')
CSV_PATH    = Path('/g/data/k10/zr7147/ORCESTRA_dropsondes_categories.csv')
IMERG_PATH  = Path('/g/data/k10/zr7147/ORCESTRA_IMERG_Combined_Cropped.nc')
OUTPUT_DIR  = Path('../outputs/figures/imerg_comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────────
print('Loading datasets...')
ds_sonde  = xr.open_zarr(str(ZARR_PATH))
df_cats   = pd.read_csv(CSV_PATH)
ds_imerg  = xr.open_dataset(str(IMERG_PATH))
imerg_times_np = imerg_times_as_np64(ds_imerg)
cum_index = build_circle_sonde_index(ds_sonde)

circles   = ds_sonde['circle'].values
cat_col   = 'category_plane' if 'category_plane' in ds_sonde else 'category_avg'
cats      = ds_sonde[cat_col].values
ctimes    = ds_sonde['circle_time'].values

# p_top per circle (for label)
omega_b = ds_sonde['omega'].values.astype(float)
p_b     = ds_sonde['p_mean'].values.astype(float)
p_tops  = []
for i in range(ds_sonde.sizes['circle']):
    valid = np.isfinite(omega_b[i]) & np.isfinite(p_b[i])
    p_tops.append(float(p_b[i, np.where(valid)[0][-1]]) if valid.sum() > 0 else np.nan)
p_tops = np.array(p_tops)

print(f'Ready — {len(circles)} circles, {len(imerg_times_np)} IMERG timesteps')

In [ ]:
# ── Plot function ─────────────────────────────────────────────────────────────
def run_plot(circle_id: int) -> None:
    ci  = int(np.where(circles == circle_id)[0][0])
    out = OUTPUT_DIR / f'circle_{circle_id:03d}.png'
    print(f'Plotting circle {circle_id}  ({cats[ci]})  p_top={p_tops[ci]/100:.1f} hPa ...')
    try:
        info = plot_one_circle(
            ds_sonde, df_cats, ds_imerg, imerg_times_np, cum_index,
            circle_id, out,
            category_col='category_plane' if 'category_plane' in df_cats.columns else 'category_avg',
        )
        print(f"  Window : {info['circle_start']} – {info['circle_end']} UTC")
        print(f"  IMERG  : {info['n_imerg_panels']} panels — {info['imerg_times']}")
        print(f"  Saved  : {info['output']}")
        display(Image(filename=str(out), width=1100))
    except Exception as e:
        print(f'  ERROR: {e}')

In [ ]:
# ── Interactive dropdown ───────────────────────────────────────────────────────
def _klaster(pt):
    if   pt < 20000: return 'A'
    elif pt < 22000: return 'B'
    else:            return 'C⚠️'

def _short_cat(c):
    if 'Top'    in c: return 'TH'
    if 'Bottom' in c: return 'BH'
    return 'IN'

options = [
    (
        f"Circle {int(c):3d} │ {_short_cat(cat)} │ {str(ct)[:10]} │ "
        f"p_top={pt/100:.0f} hPa │ Klaster {_klaster(pt)}",
        int(c)
    )
    for c, cat, ct, pt in zip(circles, cats, ctimes, p_tops)
    if np.isfinite(pt)
]

dropdown = widgets.Dropdown(
    options=options,
    value=int(circles[0]),
    description='Circle:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '60px'},
)
btn = widgets.Button(description='Plot', button_style='primary',
                     layout=widgets.Layout(width='80px'))
out_widget = widgets.Output()

def _on_btn(b):
    out_widget.clear_output(wait=True)
    with out_widget:
        run_plot(dropdown.value)

btn.on_click(_on_btn)
display(widgets.HBox([dropdown, btn]), out_widget)

# Auto-plot the first circle on load
with out_widget:
    run_plot(int(circles[0]))

In [ ]:
# ── Manual mode: set circle number here then Shift+Enter ──────────────────────
CIRCLE = 1   # ← ubah nomor circle di sini

run_plot(CIRCLE)

In [ ]:
# ── Batch: generate figures for all circles in one category ──────────────────
CATEGORY  = 'Top-Heavy'   # 'Top-Heavy' | 'Bottom-Heavy' | 'Inactive / Suppressed'
SAVE_ONLY = True          # True = save PNG only, False = also display each one

target = [int(c) for c, cat in zip(circles, cats) if CATEGORY.split()[0] in cat]
print(f'Batch: {len(target)} circles in "{CATEGORY}"')

for cid in target:
    out = OUTPUT_DIR / f'circle_{cid:03d}.png'
    ci  = int(np.where(circles == cid)[0][0])
    try:
        info = plot_one_circle(
            ds_sonde, df_cats, ds_imerg, imerg_times_np, cum_index,
            cid, out,
            category_col='category_plane' if 'category_plane' in df_cats.columns else 'category_avg',
        )
        print(f"  circle={cid:3d}  {cats[ci]:<35s}  saved → {out.name}")
        if not SAVE_ONLY:
            display(Image(filename=str(out), width=1000))
    except Exception as e:
        print(f"  circle={cid:3d}  ERROR: {e}")

print('Done.')